# Stage 3: compute-matched control run (base Qwen3-1.7B, base tokenizer)

The adapted model scored 1.718 bits per character on FLORES Urdu against the base model's 2.339, but it also had 16M tokens of Urdu training the base never had. This run separates the two: the **base model with its own tokenizer**, trained with the main run's recipe on the same Urdu Wikipedia and English replay, for the **same number of tokens** (800 steps x 32 blocks x 1,024 tokens). Decision 6 in `docs/decisions.md`.

- Same as notebook 07: LoRA rank 64 on all projections, learning rate 2e-4, warmup 50, cosine to 10%, fp16, micro batch 1, 10% English replay, seed 0, in-run eval every 100 steps on the same 200 held-out Urdu lines.
- Different by design: no new rows and no phase A, so all 800 steps train LoRA (the main run trained LoRA for 700). The base tokenizer needs about 2.29x more tokens for the same Urdu, so the control sees about 0.64 epochs of the Urdu text where the main run saw 1.47. That is what matching on compute means.

Checkpoints go to your private Hub repo every 50 steps, with a permanent milestone every 200. If the session ends, run the notebook again with the same settings and it resumes. `STOP_AT_STEP = 400` gives a shorter run (about 4.5 h) to compare with the main run's step-400 milestone; a later session with `None` continues it to 800.

Settings before running: Accelerator **GPU T4 x2** (one GPU is used), Internet **on**, Secret **HF_TOKEN** attached. **No input is needed** (the base model is downloaded). Set `HUB_REPO` to a **new private repo** in the first cell. Use *Save Version > Save & Run All (Commit)* so it keeps running with the tab closed.

Send back `control/adapt_summary.json`, `control/train_log.jsonl` and, if it ran, `control_quality.json`.

In [ ]:
import os
import time

T0 = time.time()  # session start, for the time guard before the optional evaluation
# A PRIVATE Hub repo for this run's checkpoints. It must NOT be the main run's repo: use a new name,
# e.g. "your-name/zubaan-qwen3-control". Resume refuses a main-run checkpoint anyway.
HUB_REPO = ""
MICRO_BATCH = 1          # the main run's setting (notebook 06 speed trial)
GRAD_CHECKPOINT = False  # the main run's setting
PHASE_A_STEPS = 0        # no new rows in the base model, so no phase A
PHASE_B_STEPS = 800      # same 800 steps x 32 blocks x 1,024 tokens as the main run (100 + 700)
# Do not change the lines above between sessions of the same run: a resumed run refuses a changed schedule.
STOP_AT_STEP = None      # None = all 800 steps (about 9 h). 400 = about 4.5 h, compared with the main run's step-400 milestone; a later session with None continues to 800
RUN_EVAL = True          # after the merge, score the control on the notebook 08 texts if the session has time left

# Peak memory of the main run was 13.3 GB of about 14.5: let the allocator avoid fragmentation.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
assert HUB_REPO, "set HUB_REPO: without it a crash or the 12-hour limit loses the run"
assert 32 % MICRO_BATCH == 0, "MICRO_BATCH must divide 32"
assert STOP_AT_STEP is None or 0 < STOP_AT_STEP <= PHASE_B_STEPS, STOP_AT_STEP
OUT_NAME = "control"
HUB = HUB_REPO

In [ ]:
# Install zubaan at one exact commit. Do not edit the next line except to change the commit.
COMMIT = "ad7692d"
REPO = f"git+https://github.com/Hashir-Aziz123/zubaan.git@{COMMIT}"
!pip install -q "zubaan[audit,train] @ {REPO}"
# Kaggle may already hold an older copy with the same version number, which pip keeps.
# Force our commit (dependencies are already in place).
!pip install -q --force-reinstall --no-deps "zubaan @ {REPO}"
# Kaggle ships torchao 0.10, which peft refuses ("only versions above 0.16 are supported") when it
# builds LoRA layers. We never use torchao, so remove it; peft then skips its torchao support.
!pip uninstall -y -q torchao
!zubaan version

In [ ]:
# CHECK 1: the installed code is the commit above, not a cached older copy.
# If this fails, use Run > Restart session and run the cells again from the top.
import json
from importlib import metadata

direct = json.loads(metadata.distribution("zubaan").read_text("direct_url.json"))
installed = direct["vcs_info"]["commit_id"]
print("installed commit:", installed)
assert installed.startswith(COMMIT), f"installed {installed}, wanted {COMMIT}"

import zubaan.adapt.pipeline  # noqa: F401

print("OK: code version verified")

In [ ]:
import os
import subprocess

# Quiet progress bars: they can overwhelm the notebook front end.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# CHECK 2: a GPU with about 16 GB. Training uses one GPU (fp16: a T4 and a P100 have no BF16).
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
except FileNotFoundError:
    raise SystemExit("No GPU in this session (nvidia-smi is missing). Settings > Accelerator > GPU T4 x2, then run again.") from None
assert r.returncode == 0, "no GPU found: turn on a GPU accelerator in the notebook settings"
print(r.stdout)
gpu_name, cap, mem = [x.strip() for x in r.stdout.splitlines()[0].split(",")]
assert float(mem.split()[0]) >= 14000, f"{gpu_name} has {mem}; need about 16 GB"
if float(cap) < 7.0:
    print("WARNING: compute capability", cap, "has no fp16 tensor cores; this will be slow. A T4 is better.")

import peft
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), "torch cannot see the GPU"
print("OK: GPU", gpu_name, "compute capability", cap, mem)

In [ ]:
# The Hub token is a Kaggle secret named HF_TOKEN (Add-ons > Secrets). It is never written in code.
if HUB_REPO:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hub checkpoints go to the PRIVATE repo", HUB_REPO)
else:
    print("No Hub repo: checkpoints stay in this session. A crash or the 12-hour limit loses the run.")

## 1. Base model

In [ ]:
from pathlib import Path

from huggingface_hub import HfApi, snapshot_download
from tokenizers import Tokenizer

# The base model, with its own tokenizer, downloaded outside /kaggle/working so it is not saved as output.
BASE = "Qwen/Qwen3-1.7B"
BASE_REVISION = HfApi().model_info(BASE).sha
MODEL_DIR = Path("/tmp/qwen3-1.7b-base")
snapshot_download(BASE, revision=BASE_REVISION, local_dir=str(MODEL_DIR))
print("base model:", BASE, "revision", BASE_REVISION, "->", MODEL_DIR)
assert not (MODEL_DIR / "zubaan_extension.json").exists(), "this is an extended folder, not the base"
vocab = Tokenizer.from_file(str(MODEL_DIR / "tokenizer.json")).get_vocab_size()
print("tokenizer vocabulary:", vocab)
assert vocab < 160_000, "the tokenizer has the 10,000 Urdu tokens: this must be the base tokenizer"
print("OK: base model and base tokenizer")

## 2. Configuration

In [ ]:
from pathlib import Path

import yaml

WORK = Path("/kaggle/working")
adapt = {
    "model_path": str(MODEL_DIR),
    "urdu_corpus": "data/urdu/urdu_train.txt",
    "heldout": "data/urdu/urdu_heldout.txt",
    "english_corpus": "data/english/english_replay.txt",
    "english_mb": 20,  # the base tokenizer gives about 2.3x more Urdu blocks, so 10% English needs more replay text
    "guard_urdu": ["data/flores_urd.txt", "data/flores_dev/flores_urd.txt"],
    "guard_english": ["data/flores_eng.txt", "data/flores_dev/flores_eng.txt"],
    "guard_belebele": True,
    "drop_overlaps": True,
    "data_dir": "data/adapt_control",
    "control": True,
    "block_size": 1024,
    "english_replay_ratio": 0.1,
    "output_dir": str(WORK / OUT_NAME),
    "micro_batch_size": MICRO_BATCH,
    "grad_accum": 32 // MICRO_BATCH,
    "lr_new_rows": 1e-3,
    "lr_lora": 2e-4,
    "lora_rank": 64,
    "lora_alpha": 128,
    "lora_dropout": 0.05,
    "precision": "fp16",
    "gradient_checkpointing": GRAD_CHECKPOINT,
    "device": "cuda",
    "seed": 0,
    "hub_repo": HUB,
    "hub_private": True,
    "merged_dtype": "bfloat16",
    "phase_a_steps": PHASE_A_STEPS,
    "phase_b_steps": PHASE_B_STEPS,
    "warmup_steps": 50,
    "log_every": 10,
    "eval_every": 100,
    "eval_texts": 200,
    "checkpoint_every": 50,
    "keep_checkpoints": 2,
    "milestone_every": 200,
}
config = {
    "urdu_corpus": {"raw_dir": "data/raw", "out_dir": "data/urdu", "flores": "data/flores_urd.txt", "max_mb": 100, "heldout_percent": 1.0},
    "adapt": adapt,
}
Path("adapt.yaml").write_text(yaml.safe_dump(config, sort_keys=False))
print(Path("adapt.yaml").read_text())

## 3. Data

The same Urdu and English text as notebook 07, tokenized with the base tokenizer (about 15 minutes).

In [ ]:
# Evaluation text first (FLORES devtest and dev), then the Urdu training text, then the token blocks.
# Prepare drops any training line that overlaps FLORES or Belebele text and counts it.
!zubaan data flores --out data
!zubaan data flores --split dev --out data/flores_dev
!zubaan data urdu --config adapt.yaml
!zubaan adapt prepare --config adapt.yaml

In [ ]:
meta = json.loads(Path("data/adapt_control/blocks_meta.json").read_text())
assert meta["control"] is True and meta["first_new_id"] is None, "blocks were not made for the control run"
assert meta["overlap_guard"]["hits"] == 0
urdu_tokens, n_urdu, n_en = meta["urdu"]["n_tokens"], meta["urdu"]["n_blocks"], meta["english"]["n_blocks"]
print("Urdu blocks", n_urdu, "| tokens", urdu_tokens, "| English blocks", n_en)
print("guard:", meta["overlap_guard"])
# The same Urdu text with the extended tokenizer was 16,053,605 tokens; Stage 2 measured 2.29x fewer
# tokens per character with the extension, so expect about 36.7M here.
ratio = urdu_tokens / 16053605
print(f"base / extended tokens on the same Urdu text: {ratio:.2f}x")
assert 2.0 < ratio < 2.6, "unexpected token ratio: a different corpus or tokenizer, check before training"
epoch = n_urdu / 0.9  # Urdu blocks plus 10% English
print(f"800 steps x 32 blocks = 25,600 blocks = {25600 / epoch:.2f} epochs of this data (main run: 1.47)")

## 4. Five-step check

In [ ]:
# CHECK 3: five real fp16 LoRA steps on the base model (no phase A: nothing but LoRA trains).
from zubaan.adapt.pipeline import run_adapt
from zubaan.config import load_config

cfg = load_config("adapt.yaml").adapt
assert cfg.control and cfg.phase_a_steps == 0
check = cfg.model_copy(update={
    "output_dir": WORK / "adapt_check", "hub_repo": None, "resume": False,
    "phase_b_steps": 5, "grad_accum": 2, "warmup_steps": 1,
    "log_every": 1, "eval_every": 5, "eval_texts": 20, "checkpoint_every": 5,
})
run_adapt(check, log=print)
rows = [json.loads(line) for line in (check.output_dir / "train_log.jsonl").read_text().splitlines()]
steps = [r for r in rows if "loss" in r]
skipped = sum(1 for r in steps if r["grad_norm"] == float("inf"))
print(f"steps skipped by the loss scaler: {skipped} of 5")
assert len(steps) == 5 and all(r["phase"] == "B" and r["lr_new"] is None for r in steps)
assert all(r["loss"] == r["loss"] and abs(r["loss"]) < 1e4 for r in steps)
assert skipped < 5, "every step overflowed: use precision fp32"
import shutil

shutil.rmtree(check.output_dir, ignore_errors=True)
print("OK: 5 fp16 LoRA steps, finite losses", [round(r["loss"], 3) for r in steps])

In [ ]:
import gc

import torch

gc.collect()
torch.cuda.empty_cache()
print("GPU memory in use now (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

## 5. Hub check

In [ ]:
# Before hours of training: the repo exists, is private, accepts a write, and tells us where we start.
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(HUB_REPO, private=True, exist_ok=True)
assert api.repo_info(HUB_REPO).private, f"{HUB_REPO} is public: make it private on the Hub first"
api.upload_file(path_or_fileobj=b"ok", path_in_repo="_preflight/ok.txt", repo_id=HUB_REPO)
api.delete_folder("_preflight", repo_id=HUB_REPO)
files = set(api.list_repo_files(HUB_REPO))
if "last/meta.json" in files:
    from huggingface_hub import hf_hub_download

    step = json.loads(Path(hf_hub_download(HUB_REPO, "last/meta.json")).read_text())["step"]
    print(f"OK: Hub writable. A checkpoint at step {step} exists: this run RESUMES from it.")
else:
    print("OK: Hub writable. No checkpoint yet: this run starts from step 0.")
print("milestones already saved:", sorted({f.split("/")[1] for f in files if f.startswith("milestones/")}))

# A checkpoint already in this repo must be from a control run, or the repo is the wrong one.
if "last/meta.json" in files:
    saved = json.loads(Path(hf_hub_download(HUB_REPO, "last/meta.json")).read_text())["config"]
    assert saved.get("control") is True, f"{HUB_REPO} holds a main-run checkpoint: use a new repo for the control"
    print("OK: the existing checkpoint is from this control run")

## 6. Train

Run it again with the same settings if the session ends before it reaches its target step.

In [ ]:
from zubaan.adapt.pipeline import run_adapt
from zubaan.config import load_config

cfg = load_config("adapt.yaml").adapt
summary = run_adapt(cfg, log=print, stop_at_step=STOP_AT_STEP)
print({k: summary[k] for k in ("resumed_from_step", "final_step", "total_steps", "finished")})
target = STOP_AT_STEP or summary["total_steps"]
assert summary["final_step"] == target, "the run stopped before its target: run this notebook again with the same settings to resume"
for e in summary["evals"]:
    print(e["step"], e["eval"])

## 7. Merge, save and verify

In [ ]:
# Fold LoRA into the weights (float32) and save a plain model with the base tokenizer. The merge checks
# that every embedding row and the tokenizer files are identical to the base model.
!zubaan adapt merge --config adapt.yaml
meta = json.loads((WORK / "control/model/zubaan_adapt.json").read_text())
print(json.dumps(meta, indent=1))
assert meta["control"] is True and meta["stage3"]["step"] == (STOP_AT_STEP or PHASE_B_STEPS)

## 8. Evaluate (optional, needs about 45 minutes)

In [ ]:
# Optional: score the control on the texts and protocol of notebook 08 (Stage 2 protocol: 1000 texts of at
# most 400 characters per set, all 900 Belebele questions, float32). The base model took 38 minutes in 08
# and the control has the same tokenizer, so this needs about 45 minutes.
import yaml

hours_used = (time.time() - T0) / 3600
if not RUN_EVAL:
    print("RUN_EVAL is False: evaluate later with the merged model in this notebook's output")
elif hours_used > 10.75:
    print(f"{hours_used:.1f} h used: not enough time before the 12 h limit. Evaluate in a new session from this notebook's output")
else:
    sha = json.loads(Path("data/urdu/urdu_meta.json").read_text())["files"]["heldout"]["sha256"]
    assert sha == "19c4f1b63b41b8a91cf8aa345d0f0e83945f33e403cc78c1f6f52ed7e7c34ada", "the held-out Urdu file differs from the one notebook 08 used"
    step = STOP_AT_STEP or PHASE_B_STEPS
    name = f"control-step{step}"
    evaluate = {
        "models": {name: str(WORK / "control/model")},
        "heldout": "data/urdu/urdu_heldout.txt",
        "eval_urdu": "data/flores_urd.txt",
        "eval_english": "data/flores_eng.txt",
        "quality_texts": 1000,
        "max_chars": 400,
        "belebele_questions": None,
        "dtype": "float32",
        "device": "cuda",
        "batch_size": 4,
        "max_batch_tokens": 2048,
        "output": str(WORK / "control_quality.json"),
    }
    Path("evaluate.yaml").write_text(yaml.safe_dump({"evaluate": evaluate}, sort_keys=False))
    !zubaan adapt eval --config evaluate.yaml
    row = json.loads((WORK / "control_quality.json").read_text())["variants"][name]
    print({k: round(v["bits_per_character"], 3) for k, v in row["bpc"].items()})
    print("adapted (notebook 08): urdu_heldout 1.486, urdu_flores 1.718, english_flores 1.066; Belebele urd acc_char 0.340")

## 9. Output size

In [ ]:
# Local checkpoints are on the Hub; drop them from the output, keep the merged model and the logs.
import shutil

shutil.rmtree(WORK / "control" / "checkpoints", ignore_errors=True)

total = 0
for f in sorted(WORK.rglob("*")):
    if f.is_file() and f.stat().st_size > 5e6:
        total += f.stat().st_size
        print(f"{f.stat().st_size / 1e9:6.2f} GB  {f.relative_to(WORK)}")
print(f"total of files over 5 MB: {total / 1e9:.1f} GB")
assert total < 19e9, "output is close to Kaggle's 20 GB limit: delete data/raw or old checkpoints"